In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd()

if cwd.name == "notebooks":
    BASE_DIR = cwd.parent
else:
    BASE_DIR = cwd

DATA_DIR = BASE_DIR / "data" / "raw"
TRAIN_DIR = DATA_DIR / "MINDsmall_train"
DEV_DIR = DATA_DIR / "MINDsmall_dev"

print("BASE_DIR:", BASE_DIR)
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("DEV_DIR exists:", DEV_DIR.exists())

BASE_DIR: c:\Users\rlagu\Desktop\news-recommender-project
TRAIN_DIR exists: True
DEV_DIR exists: True


In [4]:
news_cols = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]

train_news = pd.read_csv(
    TRAIN_DIR / "news.tsv",
    sep="\t",
    names=news_cols
)

dev_news = pd.read_csv(
    DEV_DIR / "news.tsv",
    sep="\t",
    names=news_cols
)

print("train_news:", train_news.shape)
print("dev_news:", dev_news.shape)

train_news.head()

train_news: (51282, 8)
dev_news: (42416, 8)


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [5]:
def parse_impressions(impression_str):
    parsed = []
    
    if pd.isna(impression_str):
        return parsed
    
    for item in impression_str.split():
        news_id, label = item.rsplit("-", 1)
        parsed.append((news_id, int(label)))
    
    return parsed


def build_samples(behaviors_df, n_rows=None):
    rows = []
    
    target_df = behaviors_df.head(n_rows) if n_rows is not None else behaviors_df
    
    for _, row in target_df.iterrows():
        history = row["history"]
        if pd.isna(history):
            history = ""
        
        for news_id, label in parse_impressions(row["impressions"]):
            rows.append({
                "impression_id": row["impression_id"],
                "user_id": row["user_id"],
                "history": history,
                "candidate_news": news_id,
                "label": label
            })
    
    return pd.DataFrame(rows)

In [6]:
news_all = pd.concat([train_news, dev_news], ignore_index=True)
news_all = news_all.drop_duplicates("news_id").reset_index(drop=True)

news_all["title"] = news_all["title"].fillna("")
news_all["abstract"] = news_all["abstract"].fillna("")
news_all["text"] = news_all["title"] + " " + news_all["abstract"]

news_all = news_all[[
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "text"
]]

print(news_all.shape)
news_all.head()

(65238, 6)


,news_id,category,subcategory,title,abstract,text
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...","The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,50 Worst Habits For Belly Fat These seemingly ...
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",I Was An NBA Wife. Here's How It Affected My M...
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...","How to Get Rid of Skin Tags, According to a De..."


In [9]:
behavior_cols = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

train_behaviors = pd.read_csv(
    TRAIN_DIR / "behaviors.tsv",
    sep="\t",
    names=behavior_cols
)

dev_behaviors = pd.read_csv(
    DEV_DIR / "behaviors.tsv",
    sep="\t",
    names=behavior_cols
)

print("train_behaviors:", train_behaviors.shape)
print("dev_behaviors:", dev_behaviors.shape)

train_behaviors.head()

train_behaviors: (156965, 5)
dev_behaviors: (73152, 5)


,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [10]:
train_samples = build_samples(train_behaviors, n_rows=5000)
dev_samples = build_samples(dev_behaviors, n_rows=5000)

print("train_samples:", train_samples.shape)
print("dev_samples:", dev_samples.shape)

dev_samples.head()

train_samples: (187715, 5)
dev_samples: (187009, 5)


,impression_id,user_id,history,candidate_news,label
0,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N28682,0
1,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N48740,0
2,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N31958,1
3,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N34130,0
4,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N6916,0


In [11]:
news_all = pd.concat([train_news, dev_news], ignore_index=True)
news_all = news_all.drop_duplicates("news_id").reset_index(drop=True)

news_all["title"] = news_all["title"].fillna("")
news_all["abstract"] = news_all["abstract"].fillna("")
news_all["text"] = news_all["title"] + " " + news_all["abstract"]

news_all = news_all[[
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "text"
]]

PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

news_all.to_csv(PROCESSED_DIR / "news_all.csv", index=False, encoding="utf-8-sig")
train_samples.to_csv(PROCESSED_DIR / "train_samples_5000.csv", index=False, encoding="utf-8-sig")
dev_samples.to_csv(PROCESSED_DIR / "dev_samples_5000.csv", index=False, encoding="utf-8-sig")

print("saved!")
print(news_all.shape)
print(train_samples.shape)
print(dev_samples.shape)

saved!
(65238, 6)
(187715, 5)
(187009, 5)
